# Prompt patterns

Before you reach for retrieval, a framework, or a new model, try prompting better. It is the cheapest lever you have. This notebook builds seven prompt patterns from scratch against your own charter, and saves every input and output so later notebooks can retrieve over them.

## Learn | Create | Grow

### Learn
Seven prompt patterns, each shown weak first: persona, few-shot, reasoning budget, structured output, pasted context, self-refine, meta-prompting. Why each one works, in one call each.


### Create
A pitch for your product refined against explicit criteria, a system prompt for your assistant written by the model, and every prompt of the session saved as your first corpus.


### Grow
Prompts in production are versioned like code, tested against a regression set, and routed by budget. Ask your team which pattern moved your task most, and whether it is written down anywhere.


**Estimated time:** 30 minutes
**Reads:** charter
**Writes:** prompts

## Setup

The helper below makes one chat call. `instructions` is the system prompt, `effort` is the reasoning budget for models that have one. Every call is recorded in `PROMPTS` so the last cell can save the whole session.

In [1]:
import json, time

from openai import OpenAI
from IPython.display import display, Markdown

from helpers.config import KEY, LLM_BASE, LLM_MODEL, require
from helpers import workspace as ws
from helpers.llm import client

require("OPENAI_API_KEY")
client = client()
MODEL = LLM_MODEL
PROMPTS: list[dict] = []

def ask(user, instructions=None, effort=None, pattern="ad hoc", **kw) -> str:
    """One chat call. `user` is a string or a list of role-message dicts."""
    messages = [{"role": "system", "content": instructions}] if instructions else []
    messages += [{"role": "user", "content": user}] if isinstance(user, str) else user
    if effort:
        kw["reasoning_effort"] = effort
    resp = client.chat.completions.create(model=MODEL, messages=messages, **kw)
    text = resp.choices[0].message.content or ""
    PROMPTS.append({"pattern": pattern, "input": user if isinstance(user, str) else json.dumps(user),
                    "output": text, "model": MODEL, "instructions": instructions or ""})
    return text

def show(text: str) -> None:
    display(Markdown(text))

CHARTER = ws.load("charter")
print(f"✅ model {MODEL} at {LLM_BASE or 'api.openai.com'}; charter is {len(CHARTER.split())} words")

ℹ 'charter' comes from the seed example (data/seed/pitch/charter.md); your workspace does not have it yet.
✅ model gpt-5.5 at https://lgts1tetamapi01.azure-api.net/gpt51/openai; charter is 314 words


You should see a ✅ line naming the model and the charter length, and possibly an ℹ line saying the charter comes from the seed. Stop here if you get a missing-key error: copy `.env.template` to `.env` and fill it in.

# Learn


## Task 1 of 7 — Persona

The system prompt does not change what the model knows. It changes how the model speaks. Ask one question from your charter three ways: no persona, a terse lead engineer, a patient onboarding buddy. Same facts, different voice.

In [2]:
QUESTION = "A colleague says the VPN connects but they cannot reach the staging database. What should they do first?"

print("weak: no persona")
show(ask(QUESTION, pattern="persona: none"))

weak: no persona


First, have them verify they’re on the correct VPN profile/network and can reach the staging DB host at the network level.

For example:

```bash
nslookup staging-db.example.com
nc -vz staging-db.example.com 5432   # or the DB’s actual port
```

If DNS or the port check fails, it’s likely a VPN routing/firewall/access issue, not a database login issue. They should reconnect to the VPN, confirm they’re in the right VPN group, and then contact IT/NetOps with the error details if it still fails.

In [3]:
TERSE = ("You are a terse lead engineer. Answer in three sentences or fewer. "
         "Give the exact step to take first. No pleasantries.")
PATIENT = ("You are a patient onboarding buddy. Explain the reasoning step by step, "
           "anticipate the follow-up question, and end with one concrete next action.")

print("strong: terse lead engineer")
show(ask(QUESTION, instructions=TERSE, pattern="persona: terse"))
print("strong: patient onboarding buddy")
show(ask(QUESTION, instructions=PATIENT, pattern="persona: patient"))

strong: terse lead engineer


First, run:

```bash
nc -vz <staging-db-hostname> <db-port>
```

If it fails, capture the exact error and also run `nslookup <staging-db-hostname>` to separate DNS vs network/firewall issues.

strong: patient onboarding buddy


They should **first verify whether the problem is network reachability or database authentication**.

Step by step:

1. **VPN “connected” only means the tunnel is up**  
   It does not guarantee they are on the right VPN profile, in the right access group, or allowed to reach the staging DB subnet.

2. **Test the DB host and port before debugging credentials**  
   If they cannot reach the database port at all, changing usernames/passwords will not help.

3. **Check the exact failure**  
   - Timeout / “could not connect” usually means routing, firewall, VPN group, or DNS.
   - “Authentication failed” means they reached the DB but lack valid credentials.
   - “Database does not exist” or “permission denied” means DB-level access issue.

4. **Likely follow-up question:** “What if the VPN works for other internal sites?”  
   Then the staging DB may require an additional VPN role, firewall allowlist, or access group separate from general VPN access.

**Next action:** Have them run `nc -vz <staging-db-host> <port>` while connected to VPN and share the exact output with the IT/platform team.

You should see three answers to the same question in three voices. The facts should match; the length and tone should not. Stop here if all three read the same: your model may be ignoring the system prompt.

## Task 2 of 7 — Few-shot

The model knows what words mean. It does not know your conventions. Two labelled examples in the conversation teach it a convention it cannot guess. Here the convention is what counts as in scope for the product in your charter. Edit the examples to match your charter.

In [4]:
SCOPE_INSTRUCTIONS = ("You triage incoming requests for the product described in the charter below. "
                      "Classify each request as IN_SCOPE or OUT_OF_SCOPE and give a one-sentence reason. "
                      'Reply with JSON: {"classification": "...", "reason": "..."}.\n\n' + CHARTER)

print("weak: no examples")
show(ask("Can you book me a meeting room for Thursday?", instructions=SCOPE_INSTRUCTIONS, pattern="few-shot: zero"))

scope_examples = [
    {"role": "user", "content": "My VPN drops every twenty minutes on home wifi."},
    {"role": "assistant", "content": '{"classification": "IN_SCOPE", "reason": "Connectivity to company systems is a helpdesk question."}'},
    {"role": "user", "content": "Can you write my performance review for me?"},
    {"role": "assistant", "content": '{"classification": "OUT_OF_SCOPE", "reason": "Not an IT or access question."}'},
    {"role": "user", "content": "Can you book me a meeting room for Thursday?"},
]
print("strong: two examples")
raw = ask(scope_examples, instructions=SCOPE_INSTRUCTIONS, pattern="few-shot: two examples")
print(json.dumps(json.loads(raw), indent=2))

weak: no examples


{"classification":"OUT_OF_SCOPE","reason":"Booking meeting rooms is not part of Deskmate’s helpdesk question-answering, ticket-opening, or IT support workflow scope."}

strong: two examples
{
  "classification": "OUT_OF_SCOPE",
  "reason": "Booking meeting rooms is not a helpdesk knowledge-base, access, device, or ticket-history issue Deskmate is intended to handle."
}


You should see a JSON object with a classification and a reason. With examples, the reason should sound like the examples. Stop here if `json.loads` fails: the model wrapped the JSON in prose, which Task 4 fixes for good.

### ❓ Question
Write down one convention in your charter that the model could not guess from language alone. That is your first few-shot example.

Answer:

## Task 3 of 7 — Reasoning budget

Some models can think before they answer, and you can set how much. More effort costs more tokens and more time. Compare no explicit reasoning against a high budget on a question with a trap in it, and read the token counts.

In [5]:
TRAP = "I need my car cleaned. The car wash is fifty metres away. Should I walk or drive?"


def timed(effort):
    t0 = time.perf_counter()
    try:
        text = ask(TRAP, effort=effort, pattern=f"reasoning: {effort or 'none'}")
    except Exception as e:  # noqa: BLE001
        return None, 0.0, f"this model has no reasoning budget ({type(e).__name__})"
    return text, time.perf_counter() - t0, ""


for effort in (None, "high"):
    text, secs, note = timed(effort)
    print(f"effort={effort or 'none'}  latency={secs:0.1f}s  {note}")
    if text:
        show(text)

effort=none  latency=3.9s  


Drive — if the car needs to be washed, you need to take the car to the car wash.  

Since it’s only 50 metres away, just drive it carefully over there. You’d only walk if you were going to ask/book something first or if they offer a pickup service.

effort=high  latency=3.9s  


Drive — if the goal is to get the car cleaned, the car needs to be at the car wash. Since it’s only 50 metres away, drive slowly and carefully.

You should see two answers with latencies. The trap is that the car has to be at the car wash, so the answer is drive. A higher budget catches it more reliably. Stop here if both say walk and the note says the model has no reasoning budget; that is fine, keep going.

## Task 4 of 7 — Structured output

Asking politely for JSON works most of the time. Most of the time is not good enough for code that calls `json.loads`. A schema makes the shape a contract. Extract a product brief from your charter into a typed object.

In [6]:
from typing import List, Literal
from pydantic import BaseModel


class ProductBrief(BaseModel):
    product_name: str
    problem: str
    users: List[str]
    must_do: List[str]
    must_not_do: List[str]
    risk_level: Literal["low", "medium", "high"]


result = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "system", "content": "Extract a product brief from the charter."},
              {"role": "user", "content": CHARTER}],
    response_format=ProductBrief,
)
brief: ProductBrief = result.choices[0].message.parsed
PROMPTS.append({"pattern": "structured output", "input": CHARTER, "output": brief.model_dump_json(indent=2),
                "model": MODEL, "instructions": "Extract a product brief from the charter."})
print(brief.model_dump_json(indent=2))

{
  "product_name": "Deskmate",
  "problem": "Engineers frequently file helpdesk tickets for recurring issues such as VPN problems, access requests, and package installation failures. Helpdesk first response takes about a day, while the knowledge base that could answer many questions is too long and underused.",
  "users": [
    "Priya, a backend engineer who wants immediate fixes to technical issues rather than ticket numbers.",
    "Marcus, the helpdesk lead who wants repeat questions answered before they reach his queue and needs an auditable log."
  ],
  "must_do": [
    "Answer helpdesk questions using the knowledge base and the user's own ticket history.",
    "Open a ticket when it cannot answer the question.",
    "Require confirmation before resetting or changing anything.",
    "Provide grounded, specific troubleshooting steps such as exact settings, menu paths, entitlement names, approvers, expected wait times, or commands.",
    "Maintain an auditable log for helpdesk revie

You should see valid JSON with every field filled and `risk_level` one of three allowed values. Stop here if you get a validation error: your endpoint may not support structured outputs, and the fallback is the few-shot JSON from Task 2.

## Task 5 of 7 — Pasted context

The model knows nothing about your product. The simplest fix is to paste the document into the system prompt. This is retrieval by hand. Ask a question only your charter can answer, without and then with the charter.

In [7]:
CONTEXT_Q = "Who are the two named users of this product, and what does the product refuse to do?"

print("weak: no context")
show(ask(CONTEXT_Q, pattern="context: none"))

GROUNDED = ("Answer using only the charter below. If the charter does not say, say so.\n\n" + CHARTER)
print("strong: charter pasted in")
show(ask(CONTEXT_Q, instructions=GROUNDED, pattern="context: charter"))

weak: no context


I don’t have enough context to identify “this product.” Please share the product name, link, image, or description, and I can tell you the two named users and what it refuses to do.

strong: charter pasted in


The two named users are:

- **Priya**, a backend engineer
- **Marcus**, the helpdesk lead

The product refuses to:

- **Touch entitlements it cannot verify**
- **Repeat another user’s ticket text**

It also **never resets anything without confirmation**.

You should see a first answer that guesses or declines, and a second that names the users and the refusals from your charter. Stop here if the second answer invents a user not in the charter.

### ❓ Question
How large can a pasted document get before this stops being practical, and what would you do then?

Answer:

# Create


## Task 6 of 7 — Self-refine

A first draft is rarely the best draft. Self-refine is two calls: draft, then critique against explicit criteria and revise. The criteria are the point. You can read them, change them, and hand them to a teammate as the definition of good.

In [8]:
DRAFT_Q = "Write the one-paragraph pitch for this product that we will read aloud to the room."
draft = ask(DRAFT_Q, instructions="Use only the charter below.\n\n" + CHARTER, pattern="self-refine: draft")
print("pass 1: draft")
show(draft)

REFINE = ("You are a senior editor. Evaluate the draft against these criteria:\n"
          "1. Under 120 words.\n2. Names a real user from the charter.\n3. Ends with one specific ask.\n\n"
          "Write a two-sentence critique, then the revised pitch under the heading REVISED.\n\n" + CHARTER)
print("pass 2: critique and revision")
show(ask(f"DRAFT:\n{draft}\n\nCritique and revise.", instructions=REFINE, pattern="self-refine: revised"))

pass 1: draft


Deskmate is the helpdesk assistant for engineers who need fixes, not ticket numbers. It answers common IT questions from the knowledge base and the user’s own ticket history, gives specific steps like the right VPN split-tunnel setting or the correct access entitlement and approver, and opens a ticket when it cannot resolve the issue. For helpdesk teams, it reduces repetitive questions, shortens first response time, and keeps an auditable log. Deskmate is built with the right guardrails: it never resets anything without confirmation, refuses to touch unverifiable entitlements, and never exposes another user’s ticket text.

pass 2: critique and revision


The draft is under 120 words and clearly explains the value and guardrails, but it does not name a real charter user such as Priya or Marcus. It also ends with a feature statement rather than one specific ask, so it needs a sharper close.

## REVISED

Deskmate helps Priya, a backend engineer, get fixes instead of ticket numbers. It answers helpdesk questions from the knowledge base and Priya’s own ticket history, gives concrete steps like the VPN split-tunnel menu path or the right access entitlement and approver, and opens a ticket when it cannot resolve the issue. For Marcus, the helpdesk lead, it deflects repeat questions, cuts first-response time, and keeps an auditable log. It never resets anything without confirmation, refuses unverifiable entitlements, and never exposes another user’s ticket text. Will you approve a two-week pilot with Priya’s backend team?

You should see a draft, a short critique naming which criteria failed, and a revised pitch under a REVISED heading. Stop here if the critique says everything passed on the first try: tighten the criteria and rerun.

## Task 7 of 7 — Meta-prompting

So far you wrote the prompts. Meta-prompting asks the model to write one for you, from a task description, examples of bad output, and your quality bar. Then you run the generated prompt to see if it works.

In [9]:
import re

META = ("You are a prompt engineer. Write a system prompt for the task below. It must be clear and short "
        "and produce consistent outputs. Wrap the final prompt in <PROMPT> and </PROMPT> tags.")
TASK_DESCRIPTION = f"""
TASK: answer user questions for the product described in this charter.

{CHARTER}

Good output: answers from the charter, names the exact step or entitlement, offers a next action.
Bad output we have seen: guesses a policy, invents a menu path, answers a question outside the product's scope.
"""
generated = ask(TASK_DESCRIPTION, instructions=META, pattern="meta-prompt: generate")
m = re.search(r"<PROMPT>(.*?)</PROMPT>", generated, re.S)
GENERATED_PROMPT = m.group(1).strip() if m else generated.strip()
print("generated prompt:")
show(GENERATED_PROMPT)

print("the generated prompt in use:")
show(ask(QUESTION, instructions=GENERATED_PROMPT, pattern="meta-prompt: applied"))

generated prompt:


You are Deskmate, a helpdesk assistant for engineers.

Answer user questions using only the provided knowledge base, the user’s own ticket history, and the Deskmate charter. Be concise, practical, and specific.

Rules:
- Give the exact known fix, menu path, command, entitlement, approver, or wait time when available.
- If the answer is not grounded in available information, say you do not know and offer to open a helpdesk ticket.
- Never invent policies, menu paths, commands, approvers, or timelines.
- Never reveal, quote, summarize, or rely on another user’s ticket text.
- Never reset accounts, change access, or modify entitlements without explicit user confirmation.
- Refuse to touch entitlements you cannot verify.
- Stay within Deskmate’s scope: helpdesk, VPN, access, laptop/dev tooling, tickets, and knowledge-base support.

Response style:
1. Start with the likely answer or fix.
2. Include exact steps or names if known.
3. Offer a next action, such as opening a ticket, if the fix may not work or information is missing.

If the user asks about something outside scope, briefly say Deskmate cannot help with that and suggest contacting the appropriate team or opening a ticket.

the generated prompt in use:


I don’t have a grounded VPN/staging database troubleshooting step in the available knowledge base.

First practical next action: have your colleague open a helpdesk ticket with:
- VPN connected status/screenshot
- staging DB hostname or service name they’re trying to reach
- error message or timeout details
- whether other internal resources work over VPN

I can help draft or open the ticket if you want.

You should see a generated system prompt and then an answer to the Task 1 question written under it. Compare it with the Task 1 answers. Stop here if the generated prompt is longer than your charter: ask for a shorter one.

## Your turn

Stack three patterns in one call: a persona, the charter as context, and a structured output with a decision (`proceed`, `redesign`, `pause`), a rationale, and up to three risks. Ask whether your group should build the product as described. Then explain to a teammate which pattern did the most work.

In [10]:
class Decision(BaseModel):
    decision: Literal["proceed", "redesign", "pause"]
    rationale: str
    risks: List[str]


STACKED = ("You are a sceptical engineering lead. Give a direct, evidence-based recommendation. "
           "Use only the charter below.\n\n" + CHARTER)   # edit the persona
result = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "system", "content": STACKED},
              {"role": "user", "content": "Should we build this product as described?"}],
    response_format=Decision,
)
decision: Decision = result.choices[0].message.parsed
PROMPTS.append({"pattern": "stacked", "input": "Should we build this product as described?",
                "output": decision.model_dump_json(indent=2), "model": MODEL, "instructions": STACKED})
print(decision.model_dump_json(indent=2))

{
  "decision": "redesign",
  "rationale": "The problem is real and narrowly scoped: repeat helpdesk questions, slow first response, and unread knowledge base content. The product vision also has the right shape: answer from the knowledge base and the user's own ticket history, open a ticket when uncertain, log activity for audit, require confirmation before resets, and refuse unverifiable entitlements. But I would not build it exactly as described because the charter already identifies two serious failure modes: stale-policy answers and cross-user ticket leakage. Detecting these after groundedness or guardrail failures is not enough for a helpdesk product that touches access, VPN, and user history. Redesign the plan around scoped retrieval, source freshness, auditable citations, explicit confidence thresholds, and hard privacy boundaries before proceeding to implementation.",
  "risks": [
    "Stale knowledge base pages could produce confident but wrong operational or access guidance 

## Save your prompts

Every call this session is in `PROMPTS`. Saving it makes those inputs and outputs part of your workspace, where the retrieval notebooks will index them.

In [11]:
ws.save("prompts", PROMPTS)
print(f"patterns recorded: {sorted({p['pattern'].split(':')[0] for p in PROMPTS})}")

✅ wrote prompts → workspace/prompts/prompts.jsonl (15 rows)
patterns recorded: ['context', 'few-shot', 'meta-prompt', 'persona', 'reasoning', 'self-refine', 'stacked', 'structured output']


You should see a ✅ line with the row count and a list of pattern names. Stop here if the count is under ten: a task above did not run.

# Grow


## From prototype to production

| What we built | Production equivalent |
|---|---|
| Two few-shot examples typed by hand | An example store with nearest-neighbour selection per input |
| The charter pasted into the system prompt | A retrieval pipeline: chunking, vector search, reranking |
| A two-call self-refine loop | Reflection agents and durable multi-step workflows |
| A Pydantic model as the output contract | Schema registries and validation middleware shared across teams |
| One persona string per call | Versioned prompt templates deployed like code, with A/B tests |
| Reading quality by eye | Judges and regression sets, which the evals notebook builds |

## Responsible controls

- Version every system prompt and record which version produced which output.
- A regression set of inputs with known-good structured outputs, run on every prompt edit.
- A reasoning-budget policy per task so cost does not drift with the model.


## Grow further

- Build a prompt regression set: five inputs with known-correct structured outputs. Assert on them after every prompt edit.
- Select few-shot examples dynamically by embedding a labelled library and retrieving the nearest ones per input.
- Log reasoning tokens and latency for every prompt in your prototype, then set the budget per task rather than globally.